# 00 Recopilacion y Transformacion de Datos

Esta libreta desarrolla la **Fase 1** del proyecto: recopilacion, extraccion, transformacion, validacion y carga de los datos climaticos. El objetivo es dejar una base confiable para las fases posteriores de EDA, inteligencia de negocios y modelado predictivo.

La fase se alinea con el documento del proyecto, que solicita identificar fuentes relevantes y ejecutar un proceso ETL antes del analisis y el modelado.

## 1. Preparacion del entorno

Importamos las librerias necesarias y conectamos el notebook con los modulos reutilizables del proyecto.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.config import (
    RAW_DATA_PATH,
    CLEANED_DATA_PATH,
    PROCESSED_DATA_PATH,
    QUALITY_REPORT_PATH,
    QUALITY_RULES_PATH,
    DATA_DICTIONARY_PATH,
)
from src.data_quality import build_quality_report, build_quality_rules_report
from src.etl import (
    FINAL_COLUMNS,
    build_data_dictionary,
    build_model_ready_data,
    load_raw_data,
    run_etl,
    transform_data,
)

pd.set_option("display.max_columns", None)

## 2. Fuente de datos

La fuente principal es el archivo `data/raw/climate_change_dataset.csv`, incluido en el repositorio. Contiene indicadores climaticos globales por pais y anio, con variables como temperatura promedio, emisiones de CO2, aumento del nivel del mar, precipitacion, poblacion, energia renovable, eventos climaticos extremos y area forestal.

In [2]:
raw_df = load_raw_data(RAW_DATA_PATH)

source_summary = pd.DataFrame([
    {
        "fuente": "Climate change dataset",
        "ubicacion": str(RAW_DATA_PATH.relative_to(ROOT)),
        "formato": "CSV",
        "granularidad": "Registro por pais y anio",
        "periodo": f"{raw_df['Year'].min()}-{raw_df['Year'].max()}",
        "registros": len(raw_df),
        "paises": raw_df["Country"].nunique(),
        "proposito_analitico": "Base para ETL, EDA, BI y modelado predictivo climatico",
    }
])

source_summary

,fuente,ubicacion,formato,granularidad,periodo,registros,paises,proposito_analitico
0,Climate change dataset,data\raw\climate_change_dataset.csv,CSV,Registro por pais y anio,2000-2023,1000,15,"Base para ETL, EDA, BI y modelado predictivo c..."


## 3. Extraccion

La funcion `load_raw_data` lee el CSV original, valida que exista y confirma que no este vacio. En esta etapa los datos se retornan sin modificar.

In [3]:
display(raw_df.head())
print(f"Filas: {raw_df.shape[0]:,}")
print(f"Columnas: {raw_df.shape[1]}")

,Year,Country,Avg Temperature (°C),CO2 Emissions (Tons/Capita),Sea Level Rise (mm),Rainfall (mm),Population,Renewable Energy (%),Extreme Weather Events,Forest Area (%)
0,2006,UK,8.9,9.3,3.1,1441,530911230,20.4,14,59.8
1,2019,USA,31.0,4.8,4.2,2407,107364344,49.2,8,31.0
2,2014,France,33.9,2.8,2.2,1241,441101758,33.3,9,35.5
3,2010,Argentina,5.9,1.8,3.2,1892,1069669579,23.7,7,17.7
4,2007,Germany,26.9,5.6,2.4,1743,124079175,12.5,4,17.4


Filas: 1,000
Columnas: 10


## 4. Diagnostico inicial

Revisamos tipos de dato, valores faltantes, duplicados, rango temporal y paises incluidos para entender la calidad inicial de la fuente.

In [4]:
diagnostic_summary = pd.DataFrame({
    "tipo_dato": raw_df.dtypes.astype(str),
    "missing_values": raw_df.isna().sum(),
    "missing_pct": (raw_df.isna().mean() * 100).round(2),
    "unique_values": raw_df.nunique(dropna=False),
})

display(diagnostic_summary)
print("Duplicados exactos:", int(raw_df.duplicated().sum()))
print("Rango de anios:", int(raw_df["Year"].min()), "-", int(raw_df["Year"].max()))
print("Paises incluidos:", raw_df["Country"].nunique())

,tipo_dato,missing_values,missing_pct,unique_values
Year,int64,0,0.0,24
Country,str,0,0.0,15
Avg Temperature (°C),float64,0,0.0,292
CO2 Emissions (Tons/Capita),float64,0,0.0,194
Sea Level Rise (mm),float64,0,0.0,41
Rainfall (mm),int64,0,0.0,799
Population,int64,0,0.0,1000
Renewable Energy (%),float64,0,0.0,407
Extreme Weather Events,int64,0,0.0,15
Forest Area (%),float64,0,0.0,473


Duplicados exactos: 0
Rango de anios: 2000 - 2023
Paises incluidos: 15


In [5]:
raw_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Year,1000.0,NaN,NaN,NaN,2011.432,7.147199,2000.0,2005.0,2012.0,2018.0,2023.0
Country,1000,15,Indonesia,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Avg Temperature (°C),1000.0,NaN,NaN,NaN,19.8831,8.542897,5.0,12.175,20.1,27.225,34.9
CO2 Emissions (Tons/Capita),1000.0,NaN,NaN,NaN,10.4258,5.614665,0.5,5.575,10.7,15.4,20.0
Sea Level Rise (mm),1000.0,NaN,NaN,NaN,3.0096,1.146081,1.0,2.0,3.0,4.0,5.0
Rainfall (mm),1000.0,NaN,NaN,NaN,1738.761,708.976616,501.0,1098.75,1726.0,2362.5,2999.0
Population,1000.0,NaN,NaN,NaN,705383046.613,409390996.884411,3660891.0,343624166.0,713116635.5,1073868037.25,1397016073.0
Renewable Energy (%),1000.0,NaN,NaN,NaN,27.3005,12.970808,5.1,16.1,27.15,38.925,50.0
Extreme Weather Events,1000.0,NaN,NaN,NaN,7.291,4.422655,0.0,3.0,8.0,11.0,14.0
Forest Area (%),1000.0,NaN,NaN,NaN,40.572,17.398998,10.1,25.6,41.15,55.8,70.0


## 5. Transformacion

La transformacion es conservadora: normaliza nombres de columnas, limpia espacios en texto, asegura tipos de dato, ordena columnas, ordena filas por pais y anio, y elimina duplicados exactos si aparecen. No se imputan datos porque la fuente no presenta valores faltantes.

In [6]:
cleaned_df = transform_data(raw_df)

display(cleaned_df.head())
display(cleaned_df.dtypes.to_frame("tipo_dato"))
print("Columnas finales:")
print(FINAL_COLUMNS)

,year,country,avg_temperature_c,co2_emissions_tons_capita,sea_level_rise_mm,rainfall_mm,population,renewable_energy_pct,extreme_weather_events,forest_area_pct
0,2000,Argentina,16.9,3.9,4.0,2047,564877556,15.5,11,18.4
1,2001,Argentina,33.2,18.2,2.9,2372,1078565697,15.7,1,39.7
2,2001,Argentina,24.8,14.3,2.8,2813,332640493,33.8,12,15.6
3,2001,Argentina,21.5,1.1,4.7,2005,597841637,31.6,2,17.7
4,2001,Argentina,8.4,16.8,2.3,2035,1334032802,12.7,3,19.8


,tipo_dato
year,int64
country,string
avg_temperature_c,float64
co2_emissions_tons_capita,float64
sea_level_rise_mm,float64
rainfall_mm,int64
population,int64
renewable_energy_pct,float64
extreme_weather_events,int64
forest_area_pct,float64


Columnas finales:
['year', 'country', 'avg_temperature_c', 'co2_emissions_tons_capita', 'sea_level_rise_mm', 'rainfall_mm', 'population', 'renewable_energy_pct', 'extreme_weather_events', 'forest_area_pct']


## 6. Validaciones de calidad

Se validan reglas basicas para que las siguientes fases puedan trabajar con confianza: anios dentro del periodo esperado, poblacion positiva, porcentajes entre 0 y 100, metricas climaticas no negativas y ausencia de duplicados exactos.

In [7]:
quality_report = build_quality_report(cleaned_df)
quality_rules = build_quality_rules_report(cleaned_df)

display(quality_report)
display(quality_rules)
print("Todas las reglas pasaron:", bool(quality_rules["passed"].all()))

,total_rows,total_columns,duplicate_exact_rows,column,dtype,missing_values,missing_pct,unique_values
0,1000,10,0,year,int64,0,0.0,24
1,1000,10,0,country,string,0,0.0,15
2,1000,10,0,avg_temperature_c,float64,0,0.0,292
3,1000,10,0,co2_emissions_tons_capita,float64,0,0.0,194
4,1000,10,0,sea_level_rise_mm,float64,0,0.0,41
5,1000,10,0,rainfall_mm,int64,0,0.0,799
6,1000,10,0,population,int64,0,0.0,1000
7,1000,10,0,renewable_energy_pct,float64,0,0.0,407
8,1000,10,0,extreme_weather_events,int64,0,0.0,15
9,1000,10,0,forest_area_pct,float64,0,0.0,473


,rule,column,passed,violation_count,min_value,max_value
0,year_between_2000_and_2023,year,True,0,2000.0,2.023000e+03
1,population_positive,population,True,0,3660891.0,1.397016e+09
2,renewable_energy_pct_between_0_and_100,renewable_energy_pct,True,0,5.1,5.000000e+01
3,forest_area_pct_between_0_and_100,forest_area_pct,True,0,10.1,7.000000e+01
4,co2_emissions_tons_capita_non_negative,co2_emissions_tons_capita,True,0,0.5,2.000000e+01
5,sea_level_rise_mm_non_negative,sea_level_rise_mm,True,0,1.0,5.000000e+00
6,rainfall_mm_non_negative,rainfall_mm,True,0,501.0,2.999000e+03
7,extreme_weather_events_non_negative,extreme_weather_events,True,0,0.0,1.400000e+01


Todas las reglas pasaron: True


## 7. Dataset procesado para fases siguientes

Ademas del dataset limpio, se genera una version procesada con variables derivadas simples para BI y analisis posterior. No se hacen transformaciones propias de machine learning como escalado, division train/test o codificacion one-hot, porque eso corresponde a la fase de modelado.

In [8]:
processed_df = build_model_ready_data(cleaned_df)

display(processed_df.head())
display(processed_df[["renewable_energy_level", "temperature_category"]].describe())

,year,country,avg_temperature_c,co2_emissions_tons_capita,sea_level_rise_mm,rainfall_mm,population,renewable_energy_pct,extreme_weather_events,forest_area_pct,emissions_total_estimated,renewable_energy_level,temperature_category
0,2000,Argentina,16.9,3.9,4.0,2047,564877556,15.5,11,18.4,2.203022e+09,baja,media
1,2001,Argentina,33.2,18.2,2.9,2372,1078565697,15.7,1,39.7,1.962990e+10,baja,alta
2,2001,Argentina,24.8,14.3,2.8,2813,332640493,33.8,12,15.6,4.756759e+09,media,alta
3,2001,Argentina,21.5,1.1,4.7,2005,597841637,31.6,2,17.7,6.576258e+08,media,media
4,2001,Argentina,8.4,16.8,2.3,2035,1334032802,12.7,3,19.8,2.241175e+10,baja,baja


,renewable_energy_level,temperature_category
count,1000,1000
unique,3,3
top,media,media
freq,346,335


## 8. Carga de resultados

Ejecutamos el ETL completo para guardar los datasets y reportes en las carpetas del proyecto.

In [9]:
cleaned_df, processed_df = run_etl()

outputs = pd.DataFrame([
    {"archivo": "Dataset limpio", "ruta": str(CLEANED_DATA_PATH.relative_to(ROOT)), "existe": CLEANED_DATA_PATH.exists()},
    {"archivo": "Dataset procesado", "ruta": str(PROCESSED_DATA_PATH.relative_to(ROOT)), "existe": PROCESSED_DATA_PATH.exists()},
    {"archivo": "Reporte de calidad", "ruta": str(QUALITY_REPORT_PATH.relative_to(ROOT)), "existe": QUALITY_REPORT_PATH.exists()},
    {"archivo": "Reglas de calidad", "ruta": str(QUALITY_RULES_PATH.relative_to(ROOT)), "existe": QUALITY_RULES_PATH.exists()},
    {"archivo": "Diccionario de datos", "ruta": str(DATA_DICTIONARY_PATH.relative_to(ROOT)), "existe": DATA_DICTIONARY_PATH.exists()},
])

outputs

,archivo,ruta,existe
0,Dataset limpio,data\cleaned\climate_change_cleaned.csv,True
1,Dataset procesado,data\processed\climate_change_model_ready.csv,True
2,Reporte de calidad,reports\tables\data_quality_report.csv,True
3,Reglas de calidad,reports\tables\data_quality_rules.csv,True
4,Diccionario de datos,reports\tables\data_dictionary.csv,True


## 9. Diccionario de datos

El diccionario resume las columnas disponibles para las fases de EDA, BI y modelado predictivo.

In [10]:
data_dictionary = build_data_dictionary()
data_dictionary

,column,source_column,stage,dtype,description
0,year,Year,"cleaned, processed",int64,Anio del registro climatico.
1,country,Country,"cleaned, processed",string,Pais analizado.
2,avg_temperature_c,Avg Temperature (°C),"cleaned, processed",float64,Temperatura promedio anual en grados Celsius.
3,co2_emissions_tons_capita,CO2 Emissions (Tons/Capita),"cleaned, processed",float64,Emisiones de CO2 en toneladas por habitante.
4,sea_level_rise_mm,Sea Level Rise (mm),"cleaned, processed",float64,Aumento estimado del nivel del mar en milimetros.
5,rainfall_mm,Rainfall (mm),"cleaned, processed",int64,Precipitacion anual en milimetros.
6,population,Population,"cleaned, processed",int64,Poblacion asociada al registro.
7,renewable_energy_pct,Renewable Energy (%),"cleaned, processed",float64,Participacion de energia renovable en porcentaje.
8,extreme_weather_events,Extreme Weather Events,"cleaned, processed",int64,Cantidad de eventos climaticos extremos regist...
9,forest_area_pct,Forest Area (%),"cleaned, processed",float64,Porcentaje de area forestal.


## 10. Cierre de la fase

La fase 1 deja como resultado un proceso ETL reproducible, un dataset limpio, un dataset procesado, reportes de calidad y un diccionario de datos. Las siguientes fases pueden usar `data/cleaned/climate_change_cleaned.csv` para EDA y `data/processed/climate_change_model_ready.csv` para BI y modelado, manteniendo el split, escalado y entrenamiento dentro de la fase de aprendizaje computacional.